In [ ]:
!pip install -q langchain langchain-groq

In [ ]:
import os
from google.colab import userdata

os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")

from langchain_groq import ChatGroq

llm = ChatGroq(
    model="openai/gpt-oss-120b",
    temperature=0.3
)

In [ ]:
SYSTEM = """ROLE: You are the friendly customer assistant for ReadMore Online Bookstore.

FACTS:
We sell fiction, non-fiction, academic, and children's books.
Orders depend on stock availability.
Standard and express shipping are available.
Customers can request book recommendations based on their interests.

RULES:
Recommend books based on customer interests.
Never invent book prices or stock availability.
Ask questions if book details are unclear.
Never guarantee delivery dates.
If you don't know something, say "I'll check with the store."
Provide accurate order and return information.
Do not claim an order has been placed without confirmation.

FORMAT:
Reply politely in under 60 words.
Use friendly English or Telugu matching the customer."""

In [ ]:
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.output_parsers import StrOutputParser

prompt = ChatPromptTemplate.from_messages([
    ("system", SYSTEM),
    MessagesPlaceholder("history"),
    ("human", "{question}"),
])

chain = prompt | llm | StrOutputParser()

In [ ]:
history = []

def ask(question: str) -> str:

    answer = chain.invoke({
        "history": history,
        "question": question
    })

    history.extend([
        ("human", question),
        ("ai", answer)
    ])

    return answer

In [ ]:
print(ask("Hi, I'm Ravi. Can you recommend a book?"))

print(ask("What is my name?"))

history

In [ ]:
import csv

def run_tests(path: str = "/content/test.csv") -> None:

    rows = list(csv.DictReader(open(path)))

    passed = 0

    for row in rows:

        answer = chain.invoke({
            "history": [],
            "question": row["question"]
        })

        ok = row["must_include"].lower() in answer.lower()

        passed += ok

        print(
            "PASS" if ok else "FAIL",
            "|", row["question"],
            "→", answer[:70]
        )

    print(f"\n{passed}/{len(rows)} passed ({passed / len(rows):.0%})")

In [ ]:
import pandas as pd

data = {
    "question": [
        "What is the price of Atomic Habits?",
        "What is the price of The Alchemist?",
        "Do you sell children's books?",
        "What types of books do you sell?",
        "Can I get express shipping?"
    ],

    "must_include": [
        "check",
        "check",
        "children",
        "fiction",
        "express"
    ]
}

df = pd.DataFrame(data)

df.to_csv("/content/test.csv", index=False)

print(df)

In [ ]:
run_tests()

In [ ]:
from langchain.agents import create_agent

agent_order = create_agent(
    model=llm,
    tools=[],
    system_prompt=SYSTEM
)

result_order = agent_order.invoke({
    "messages": [
        {
            "role": "user",
            "content": "Can you recommend a book for students?"
        }
    ]
})

print(result_order["messages"][-1].content)

In [ ]:
print(ask("What is the price of Harry Potter?"))

print(ask("I like motivational books. Suggest one."))

print(ask("What is my name?"))

print(ask("Do you guarantee delivery tomorrow?"))

print(ask("Do you sell academic books?"))

print(ask("Can I return a book if I don't like it?"))